# [CVPR 2026] The Devil is in Attention Sharing: Improving Complex Non-rigid Image Editing Faithfulness via Attention Synergy

This notebook demonstrates how to use SynPS to generate edited image pairs from source and target text prompts.

**Paper:** [arXiv:2512.14423](https://arxiv.org/abs/2512.14423)  
**Project Page:** [synps26.github.io](https://synps26.github.io/)

## Import Libraries

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

import re
import math
import time
from dataclasses import dataclass

import torch
import numpy as np
from einops import rearrange
from PIL import ExifTags, Image
import matplotlib.pyplot as plt

from flux.sampling import denoise, get_schedule, prepare, unpack
from flux.util import configs, embed_watermark, load_ae, load_clip, load_flow_model, load_t5

## Configuration

Adjust the parameters below to control generation behavior.

In [ ]:
@dataclass
class Config:
    name: str = "flux-dev"          # Model name
    guidance: float = 3.5            # Classifier-free guidance scale
    num_steps: int = 50              # Number of denoising steps
    pe_threshold_max: float = 1.0    # Upper threshold for PE weight scheduling
    pe_threshold_min: float = 0.9    # Lower threshold for PE weight scheduling
    seeds: list = None               # Random seeds for generation
    output_dir: str = "./results"    # Output directory
    offload: bool = True           # Offload models to CPU to save VRAM

config = Config()
config.seeds = [0, 1]  # Generate with 2 seeds; add more as needed

## Load Models

In [ ]:
print("Loading models...")
start_time = time.time()
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

t5 = load_t5(torch_device, max_length=256 if config.name == "flux-schnell" else 512)
clip = load_clip(torch_device)
model = load_flow_model(config.name, device="cpu" if config.offload else torch_device)
ae = load_ae(config.name, device="cpu" if config.offload else torch_device)

print(f"Models loaded in {time.time() - start_time:.2f}s")

## Example Prompts

Define source and target prompt pairs below. Each source prompt is paired with one or more target prompts describing the desired edit.

### Example 1: Border Collie


In [ ]:
source_prompts = [
    "A medium-sized, long-haired black and white Border Collie is standing in the center of the square, with people passing by around it."
]
target_prompts = [
    "A medium-sized, long-haired black and white Border Collie is sitting in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is sitting in the center of the square, raising one paw to greet, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is jumping in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is looking up at the sky in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is eating dog food in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is lowering its head to eat dog food in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is jumping up to catch a frisbee in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is holding a frisbee in its mouth in the center of the square, with people passing by around it.",
    "A medium-sized, long-haired black and white Border Collie is playing on a skateboard in the center of the square, with people passing by around it."
]

## Generation Function

In [ ]:
@dataclass
class SamplingOptions:
    source_prompt: str
    target_prompt: str
    width: int
    height: int
    num_steps: int
    guidance: float
    seed: int | None


@torch.inference_mode()
def generate(
    source_prompt: str,
    target_prompt: str,
    seed: int,
    config: Config,
    t5=t5, clip=clip, model=model, ae=ae,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
):
    torch.set_grad_enabled(False)
    name = config.name
    num_steps = config.num_steps
    guidance = config.guidance
    offload = config.offload

    if name not in configs:
        available = ", ".join(configs.keys())
        raise ValueError(f"Got unknown model name: {name}, chose from {available}")

    if num_steps is None:
        num_steps = 4 if name == "flux-schnell" else 25

    torch_device = torch.device(device)
    height, width = 512, 512

    if offload:
        model.cpu()
        torch.cuda.empty_cache()
        ae.encoder.to(torch_device)

    rng = torch.Generator(device="cpu")
    opts = SamplingOptions(
        source_prompt=source_prompt,
        target_prompt=target_prompt,
        width=width,
        height=height,
        num_steps=num_steps,
        guidance=guidance,
        seed=seed,
    )

    if opts.seed is None:
        opts.seed = rng.seed()
    print(f"Generating with seed {opts.seed}:\nSource: {opts.source_prompt}\nTarget: {opts.target_prompt}")
    t0 = time.perf_counter()

    torch.manual_seed(opts.seed)
    latent_height = 2 * math.ceil(height / 16)
    latent_width = 2 * math.ceil(width / 16)
    initial_noise = torch.randn(
        (1, 16, latent_height, latent_width),
        device=torch_device, dtype=torch.bfloat16
    )

    opts.seed = None
    if offload:
        ae = ae.cpu()
        torch.cuda.empty_cache()
        t5, clip = t5.to(torch_device), clip.to(torch_device)

    info = {
        'pe_threshold_max': config.pe_threshold_max,
        'pe_threshold_min': config.pe_threshold_min,
    }
    prompts = [opts.source_prompt, opts.target_prompt]

    inp = prepare(t5, clip, initial_noise, prompt=prompts)
    timesteps = get_schedule(opts.num_steps, inp["img"].shape[1], shift=(name != "flux-schnell"))

    if offload:
        t5, clip = t5.cpu(), clip.cpu()
        torch.cuda.empty_cache()
        model = model.to(torch_device)

    print("Running batched generation for source and target prompts...")
    x = denoise(model, **inp, timesteps=timesteps, guidance=guidance, info=info)

    if offload:
        model.cpu()
        torch.cuda.empty_cache()
        ae.decoder.to(x.device)

    batch_x = unpack(x.float(), opts.height, opts.width)
    with torch.autocast(device_type=torch_device.type, dtype=torch.bfloat16):
        decoded_batch = ae.decode(batch_x)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    decoded_batch = decoded_batch.clamp(-1, 1)
    decoded_batch = embed_watermark(decoded_batch.float())
    images_np = (127.5 * (rearrange(decoded_batch, "b c h w -> b h w c") + 1.0)).cpu().byte().numpy()

    source_img = Image.fromarray(images_np[0])
    target_img = Image.fromarray(images_np[1])

    print(f"Done in {t1 - t0:.1f}s.")
    return source_img, target_img

## Generate Images

In [ ]:
all_results = []

for seed in config.seeds:
    for source_prompt in source_prompts:
        for target_prompt in target_prompts:
            source_img, target_img = generate(
                source_prompt=source_prompt,
                target_prompt=target_prompt,
                seed=seed,
                config=config,
            )
            all_results.append({
                "seed": seed,
                "source_prompt": source_prompt,
                "target_prompt": target_prompt,
                "source_img": source_img,
                "target_img": target_img,
            })

            # Save results
            save_dir = os.path.join(config.output_dir, f"seed_{seed}")
            os.makedirs(save_dir, exist_ok=True)

            # Stitch source and target side by side
            w, h = source_img.size
            combined = Image.new("RGB", (w * 2, h))
            combined.paste(source_img, (0, 0))
            combined.paste(target_img, (w, 0))

            exif_data = Image.Exif()
            exif_data[ExifTags.Base.Software] = "AI generated;txt2img;flux"
            exif_data[ExifTags.Base.Make] = "Black Forest Labs"
            exif_data[ExifTags.Base.Model] = config.name
            exif_data[ExifTags.Base.ImageDescription] = f"Source: {source_prompt}\nTarget: {target_prompt}"

            fn = os.path.join(save_dir, f"{target_prompt}.jpg")
            combined.save(fn, exif=exif_data, quality=95, subsampling=0)
            print(f"Saved: {fn}")

print(f"\nGeneration complete! {len(all_results)} image pairs generated.")

## Visualize Results

In [ ]:
for i, result in enumerate(all_results):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(result["source_img"])
    axes[0].set_title("Source", fontsize=10)
    axes[0].axis("off")
    axes[1].imshow(result["target_img"])
    axes[1].set_title("Target", fontsize=10)
    axes[1].axis("off")
    fig.suptitle(f"Seed {result['seed']}\nTarget: {result['target_prompt'][:80]}...", fontsize=9)
    plt.tight_layout()
    plt.show()